# BM25 Section Keyword Extraction

This notebook loads one of the existing BM25 pickle models from `output/` and extracts the most important terms for each section using BM25 document-side weights.

It expects a matching `*_complete.json` file for section text blocks.

In [ ]:
from __future__ import annotations

import json
import math
import pickle
import re
import string
from collections import Counter, defaultdict
from pathlib import Path

try:
    import pandas as pd
except Exception:
    pd = None

PUNCT = re.compile(r'[' + re.escape(string.punctuation) + r']')

def tokenize(text: str) -> list[str]:
    text = (text or '').lower()
    text = PUNCT.sub(' ', text)
    return [t for t in text.split() if t]

# Resolve project root whether notebook is run from project root or playground/.
cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'playground' else cwd
OUTPUT_DIR = ROOT / 'output'
INPUT_DIR = ROOT / 'input'

bm25_files = sorted(OUTPUT_DIR.glob('*_bm25.pkl'))
if not bm25_files:
    raise FileNotFoundError(f'No *_bm25.pkl files found in {OUTPUT_DIR}')

print('Available BM25 models:')
for i, p in enumerate(bm25_files, start=1):
    print(f'{i}. {p.name}')

# Pick one BM25 model (change index if needed).
MODEL_INDEX = 0
bm25_path = bm25_files[MODEL_INDEX]
doc_id = bm25_path.name.replace('_bm25.pkl', '')
candidate_complete_paths = [
    OUTPUT_DIR / f'{doc_id}_complete.json',
    INPUT_DIR / f'{doc_id}_complete.json',
]
complete_path = next((p for p in candidate_complete_paths if p.exists()), candidate_complete_paths[0])

print(f'\nSelected model: {bm25_path.name}')
print(f'Resolved section text file: {complete_path}')
if not complete_path.exists():
    raise FileNotFoundError(
        f'Matching file not found: {complete_path}. '
        'Needed for section-wise text aggregation.'
    )

In [ ]:
with bm25_path.open('rb') as f:
    state = pickle.load(f)

required_keys = {'vocab', 'idf', 'avgdl', 'k1', 'b', 'fitted'}
missing = required_keys - set(state.keys())
if missing:
    raise ValueError(f'Unexpected BM25 state format. Missing keys: {missing}')

if not state.get('fitted', False):
    raise ValueError('Loaded BM25 model is not fitted.')

vocab = state['vocab']                # token -> index
idf_table = state['idf']              # index -> idf
avgdl = float(state['avgdl'])
k1 = float(state['k1'])
b = float(state['b'])
idx_to_token = {idx: tok for tok, idx in vocab.items()}

complete = json.loads(complete_path.read_text())
text_blocks = complete['extracted_elements']['text_blocks']

# Aggregate all text by section title.
section_texts: dict[str, list[str]] = defaultdict(list)
for block in text_blocks:
    section = (block.get('section_title') or '').strip()
    text = (block.get('text') or '').strip()
    if not text:
        continue
    if not section:
        section = 'Unknown'
    section_texts[section].append(text)

section_docs = {k: '\n'.join(v) for k, v in section_texts.items()}
print('Sections found:', len(section_docs))
print('Sample sections:', list(section_docs.keys())[:8])

STOPWORDS = {
    'the', 'and', 'of', 'to', 'in', 'a', 'for', 'is', 'on', 'with', 'that', 'as', 'by',
    'are', 'be', 'this', 'we', 'an', 'or', 'from', 'at', 'our', 'it', 'can', 'which',
    'let', 'such', 'their', 'these', 'then', 'have', 'has', 'also', 'than', 'into',
    'under', 'using', 'used', 'use', 'not', 'if', 'any', 'all', 'over', 'each', 'both',
}

def bm25_term_scores_for_doc(text: str) -> dict[str, dict[str, float]]:
    """Return per-term BM25 weights for one section text."""
    tokens = tokenize(text)
    if not tokens:
        return {}

    tf = Counter(tokens)
    doc_len = len(tokens)
    norm = k1 * (1 - b + b * doc_len / max(avgdl, 1e-9))

    scores: dict[str, dict[str, float]] = {}
    for token, freq in tf.items():
        if len(token) < 3:
            continue
        if token.isdigit():
            continue
        if token in STOPWORDS:
            continue

        idx = vocab.get(token)
        if idx is None:
            continue

        idf = float(idf_table.get(idx, 0.0))
        if idf <= 0:
            continue

        score = idf * ((freq * (k1 + 1)) / (freq + norm))
        scores[token] = {'score': score, 'tf': float(freq), 'idf': idf}

    return scores

def top_terms_per_section(section_docs: dict[str, str], top_k: int = 15) -> dict[str, list[dict[str, float]]]:
    results = {}
    for section, text in section_docs.items():
        term_map = bm25_term_scores_for_doc(text)
        ranked = sorted(term_map.items(), key=lambda x: x[1]['score'], reverse=True)[:top_k]
        results[section] = [
            {'term': term, 'score': vals['score'], 'tf': vals['tf'], 'idf': vals['idf']}
            for term, vals in ranked
        ]
    return results

In [ ]:
TOP_K = 12
keywords_by_section = top_terms_per_section(section_docs, top_k=TOP_K)

# Pretty print
for section, rows in keywords_by_section.items():
    if not rows:
        continue
    print(f'\n## {section}')
    print(', '.join(r['term'] for r in rows))

# Optional tabular output
flat_rows = []
for section, rows in keywords_by_section.items():
    for rank, row in enumerate(rows, start=1):
        flat_rows.append({
            'section': section,
            'rank': rank,
            'term': row['term'],
            'bm25_score': round(row['score'], 6),
            'tf': row['tf'],
            'idf': round(row['idf'], 6),
        })

if pd is not None:
    df = pd.DataFrame(flat_rows).sort_values(['section', 'rank']).reset_index(drop=True)
    display(df.head(50))

# Save outputs for reuse
out_json = OUTPUT_DIR / f'{doc_id}_section_keywords_bm25.json'
out_json.write_text(json.dumps(keywords_by_section, indent=2, ensure_ascii=False))
print(f'\nSaved section keywords JSON to: {out_json}')

if pd is not None:
    out_csv = OUTPUT_DIR / f'{doc_id}_section_keywords_bm25.csv'
    df.to_csv(out_csv, index=False)
    print(f'Saved section keywords CSV to: {out_csv}')